<div style="width: 100%; text-align: center; margin-bottom: 20px;">
  <img src="images/cabecalho_branco.jpg" alt="Cabeçalho" style="max-width: 100%; height: auto;">
</div>

# A Regularização tipo L1 e L2 aplicada à <br> Regressão Linear

**Autor**: Carlos Gabriel de Oliveira Campos

## Uma breve introdução à Regressão Linear

A Regressão Linear é um algoritmo clássico da estatística que opera ao assumir uma relação linear entre uma ou mais variáveis independentes e uma variável dependente. Em aprendizado de máquina, esse mecanismo pode ser utilizado para *induzir* (criar) modelos preditivos cujo *target* (variável alvo) seja quantitaivo e contínuo, sendo útil para as mais diversas áreas, como:
 
- Imóveis: Prever preços de propriedades usando localização, tamanho e outros fatores.
- Finanças: A previsão de preços de ações usando taxas de juros e dados de inflação.
- Agricultura: Estimar a produção de culturas a partir de chuva, temperatura e qualidade do solo.
- E-commerce: Analisar como preço, promoções e estações afetam as vendas [1].

## Regressão Linear Simples

Para casos de uma única variável independente, chama-se Regressão Linear Simples a relação matemática entre dados que  pode ser representada pela fórmula **1.1**, em que as matrizes-coluna, $Y$ e $X$, representam a resposta quantitativa e a variável de previsão, respectivamente. $\beta_0$ e $\beta_1$ são os coeficientes, sendo o primeiro o *intercepto* - o valor estimado para $Y$ quando todas as variáveis independentes valem 0-, e $\beta_1$, o termo que ajusta as variáveis de X ao valor estimado. Ademais, o termo ϵ simboliza o erro aleatório, com média 0, e simboliza variações naturais e irredutíveis que impedem o modelo de prever com total precisão.


$$Y = \beta_0 + \beta_1 X + ϵ \tag{1.1}$$

### Calculando os coeficientes

Dado esse modelo, como podemos estimar seus coeficientes? O meio mais prático para essa tarefa é o *Método dos Quadrados Mínimos* (MQM). Para isso, é necessário o conceito de *RSS*, do inglês, *soma dos resíduos quadráticos*. Ela funciona como um indicador da precisão do modelo, uma *função de custo*! Sendo que uma menor RSS implica em uma maior precisão em relação aos dados de treino. O quadrado da diferença entre o real e o esperado é para que erros de previsão "para mais" e "para menos" não se cancelem, da mesma forma que penaliza mais severamente erros maiores, enquanto suaviza erros menores. 

$$RSS =(y_1 − \hat{\beta}_0 −\hat{\beta}_1x_1)^2+(y_2 −\hat{\beta}_0 −\hat{\beta}_1x_2)^2 + ··· + (y_n − \hat{\beta}_0 − \hat{\beta}_1x_n)^2 \tag{1.2}$$

Quando aplicamos a derivada parcial em relação a cada coeficiente, podemos encontrar que:

$$\hat{\beta}_1 = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^n (x_i - \bar{x})^2}$$

$$\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x} \tag{1.3}$$



<figure align="center">
  <img src="../../../../../../../../C:/Users/carlos2610010/OneDrive%20-%20ILUM%20ESCOLA%20DE%20CI%C3%8ANCIA/VS%20CODE/2%C2%BA%20SEM/AP.%20MAQ/esfinge/images/rss-graphic.png" alt="Representação gráfica da RSS" width="80%">
  <figcaption><em><b>Figura 1:</b> Representação em contorno e tridimensional da RSS em função dos parâmetros, em que o ponto vermelho corresponde ao par de parâmetros que minimizam a RSS para um certo conjunto de dados. [ISLP - p. 73].</em></figcaption>
</figure>


Notamos, assim, que há uma dependência de $\beta_0$ em relação a $\beta_1$, o que não é favorável aos cálculos para estimar o valor dos coeficientes, visto que qualquer mudança em $\beta_1$ irá alterar $\beta_0$. Logo, é interessante centralizar $X$, o que torna a méida $\tilde{\bar{x}} = 0$.

$$\tilde{x} = x_i - \bar{x}$$
$$\bar{\tilde{x}} = 0$$

Logo:

$$\hat{\beta}_0 = \bar{y}$$

Portanto, podemos reformular a equação 1.2 para:

$$RSS = (y_1 − \beta_0 - \hat{\beta}_1 \tilde{x_1})^2 + (y_2−\beta_0 -\hat{\beta}_1\tilde{x_2})^2 + ··· + (y_n - \beta_0 − \hat{\beta}_1\tilde{x_n})^2$$
$$RSS = \sum_{i=1}^{n} (y_i - \beta_0 - \hat{\beta}_1 \tilde{x_i})^2 \tag{1.4}$$


Agora que definimos nossa função de custo, devemos buscar como devem ser o estimadores de coeficiente $\hat{\beta_0}$ e $\hat{\beta_1}$ que minimizam a RSS. Ou seja, basta calcular as derivadas parciais da RSS em relação aos estimadores e igualar a 0, obtendo:


$$\tag{1.5}
\hat{\beta} = (\tilde{X}^T \tilde{X})^{-1} \tilde{X}^T Y$$

## Colocando em prática: Verificando a relação entre vendas e diferentes meios de publicidade

### Imports de módulos utilizados

In [76]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score



### Carregando dados sobre investimento em publicidade para prever o número de vendas

A tabela `Advertising` trata de quantas milhares de unidades de um produto foram vendidas (`sales`) dado uma quantidade em milhar de dólares ($1_000.00) de investimento para cada meio de comunicação para um mercado específico, sendo que estão disponíveis para anúncios `TV`, `radio` e `jornal`. 

Dessa forma, a primeira observação da tabela, `[230.1, 37.8, 69.2, 22.1]`, indica que em um dado mercado, investiu-se em propagandas - em dólares - 230.1 mil na TV, 37.1 mil nas rádios e 69.2 mil em matérias de jornal, resultando em 22.1 mil unidades vendidas.

In [67]:
# Criando DataFrame do pandas
df = pd.read_csv("Advertising.csv", index_col=0);

### Divisão Treino-Teste pelo método *Holdout*

As observações serão particionadas entre dados para treinamento do modelo e dados para teste do modelo.
Como há apenas 200 observações (um número relativamente baixo), serão reservados 20% dos dados para teste, ao invés dos costumeiros 10% para escalas maiores.

In [6]:
TAMANHO_TESTE = 0.2
SEMENTE = 42 

indices = df.index
indices_treino, indices_teste = train_test_split(
    indices, test_size=TAMANHO_TESTE,
    random_state=SEMENTE
)

df_treino = df.loc[indices_treino]
df_teste = df.loc[indices_teste]

### Treinamento do modelo

Como será visto em breve, há módulos em python dedicados e bem simples com ferramentas para regressão linear, mas primeiro é necessário pegar a lógica *por debaixo dos panos*.

In [7]:
# Atribuindo cada conjunto a uma Série
x_treino = df_treino["TV"]
y_treino = df_treino["vendas"]

x_teste = df_teste["TV"]
y_teste = df_teste["vendas"]

# Obtendo médias de X
media_x_treino = x_treino.mean()
media_x_teste = x_teste.mean()

# Centralizando dados da TV
x_treino_c = x_treino - media_x_treino
x_teste_c = x_teste - media_x_teste

# Calculando os parâmetro
b1 = x_treino_c.dot(y_treino) / x_treino_c.dot(x_treino_c)
b0 = y_treino.mean()

# Calcula linha de previsão do modelo
previsao = b0 + x_teste_c * b1

### Visualizando a reta ajustada

<p align="center">
  <img src="images/rls_tv_vendas.png"/>
</p>

### O quão forte é a relação entre o atributo e o alvo?

Nas estatísticas, existe uma métrica muito relevante para métodos lineares que nos ajuda a compreender a proporção de variabilidade que é explicada pelo modelo. Em outras palavras, *o quanto de $Y$ que pode ser explicado por $X$*. Assim, o **coeficiente de determinação**, ou como também é conhecido, $R^2$, é calculado como:

$$R^2 = \frac{TSS - RSS}{TSS} = 1 - \frac{RSS}{TSS}$$

Dado que a *soma total dos quadrados* (TSS) é dada por

$$ TSS = \sum_{i=1}^{n} (y_i - \bar{y})^2$$

Uma medida de $R^2$ próxima de 1 significa que uma vasta proporção da variabilidade dos valores da resposta são explicadas pela regressão, enquanto um resultado próximo de 0 implica que pouco é explicado pela regressão. Isso pode decorrer por um modelo linear errado, por uma variância de erro muito alta, ou por ambos.

In [9]:
r_quad_rls: float = r2_score(
    y_true=y_teste,
    y_pred=previsao
    )
print(f"R² aferido para dados de teste com RLS: {r_quad_rls:.2f}")

R² aferido para dados de teste com RLS: 0.67


O $R^2$ obtido de 0.67 indica que apenas a informação de `TV` é capaz de explicar dois terços da variância de $Y$, o que não é um resultado tão satisfatório, trazendo as implicações recém-comentadas, mas demonstra tamanha a influência dessa mídia para as vendas.


### O quão preciso foi o modelo? Analisando com *RMSE*

De nada adianta treinarmos um modelo, realizar as previsões, mas não medir seu desempenho. Daí surge a raíz do erro quadrático médio, do inglês *root mean squared error* (RMSE). Ele é uma abrodagem amplamente utilizada por expressar a mesma unidade da variável alvo (nesse caso, unidades vendidas). Por conta de seu fator quadrático, erros maiores são penalizados mais severamente e erros menores são suavizados, o que não o torna a medida mais direta para comunicar o erro médio - como o *erro médio absoluto* (MAE) da equação 1.7 [2] -, mas potencializa comparações entre modelos.

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
\tag{1.6}
$$

$$
MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y_i}| 
\tag{1.7}
$$

Calculando o RMSE:

In [10]:
rmse_rls: float = root_mean_squared_error(
    y_true=y_teste, 
    y_pred=previsao
    )
print(f"Raíz do erro quadrático médio (RMSE) com Regressão Linear Simples: {rmse_rls:.2f}")

Raíz do erro quadrático médio (RMSE) com Regressão Linear Simples: 3.20


Ou seja, o modelo de regressão linear simples obteve um RMSE de 3.20 unidades vendidas. 
Dessa forma, pode-se perceber como a publicidade na televisão é um preditor muito interessante para o número de vendas do produto. Ao mesmo tempo, é essencial reconhecer sua insuficiência: ele faz uma análise geral com base em apenas uma única variável, quando, na realidade, há muitos outros fatores que podem influenciar as vendas, como outros meios de publicidade, a imagem pública sobre a marca, etc...

É sempre importante ponderar qual modelo é o mais adequado a cada cenário, e tendo agora meios para medir a qualidade do modelo, por que não compará-lo a outro? E se... e se analisarmos mais de um atributo para um mesmo target!? Talvez nossa previsão fique melhor...

## Regressão Linear Múltipla

Como foi sugerido, a Regressão Linear Múltipla (RLM) emerge para induzir modelos com base em 2 *ou mais* preditores simultaneamente. Isso nos permite ter uma compreensão mais profunda das associações entre as variáveis em consideração, mas também requer um novo leque de sensibilidades para trabalhar adequadamente com a nova ferramenta. A base estatística é similar à de Regressão Linear Simples, sendo que a resposta assume a forma:
$$
\tag{2.1}
Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_p X_p 
$$




Nesse caso, $\hat{\beta}$ simboliza a matriz-coluna dos $p$ coeficientes $(\hat{\beta_1}, \hat{\beta_2}, ...,\hat{\beta_p})^T$ e $X$ é a matriz $n$ x $p$ com $n$ elementos pertencentes a cada um dos $p$ atributos.  
$$
\tag{2.2}
\hat{\beta} = (\tilde{X}^T \tilde{X})^{-1} \tilde{X}^T {Y} 
$$

### Implementação de RLM com `sci-kit`: treinamento

In [81]:
# Atribuindo Pandas/Séries a cada conjunto
X_treino_rlm = df_treino[["TV", "radio"]]
X_teste_rlm = df_teste[["TV", "radio"]]

# Instanciando modelo
modelo_rlm = LinearRegression()

# Treinando modelo
modelo_rlm.fit(X_treino_rlm, y_treino)

# Gerando previsão
previsao_rlm = modelo_rlm.predict(X_teste_rlm)

### Visualização 3D da superfície de regressão e do desempenho do modelo


<p align="center">
  <img src="images/sup_reg_3d.png" height="480" style="margin-right: 30px;" />
  <img src="images/desempenho_rlm.png" height="480" />
</p>

### Avaliando o modelo de RLM

Olha só! Agora vemos em 3D! Com essa visualização, podemos notar que $\hat{f}(X)$ - o valor previsto - assume um maior valor seguindo a linha reta entre os eixos de Radio e de TV. Isso pode ser compreendido porque ela consiste dos pontos em que o investimento em ambas as modalidades de anúncios são maximizados. Logo, as vendas também serão maximizadas. Sob outro ponto de vista, dado um orçamento, é notório que deve-se investir mais em propagandas televisivas do que na rádio, uma vez que as primeiras apresentam um maior retorno anúncio-vendas. 

Além disso, o gráfico sobre o desemepenho do modelo nos oferece uma observação comparativa com o outro gráfico. Pode-se notar como a previsão possui erros maiores (mas relativamente pequenos!) no início da reta, mas fica substancialmente mais preciso para maiores valores de `vendas`.

Feita a regressão, torna-se possível verificar as métricas de seu desempenho por meio de $R^2$ e do $RMSE$.

<table style="width: 100%; border-collapse: collapse; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); border: 1px solid #cbd5e1;">
  <thead>
    <tr style="background-color: #0f172a; color: #ffffff; text-align: left; font-size: 14px;">
      <th style="padding: 12px 16px; border: 1px solid #334155;">Métrica de Avaliação</th>
      <th style="padding: 12px 16px; text-align: center; border: 1px solid #334155;">RL Simples</th>
      <th style="padding: 12px 16px; text-align: center; border: 1px solid #334155;">RL Múltipla</th>
      <th style="padding: 12px 16px; text-align: center; border: 1px solid #334155;">Variação (Δ)</th>
      <th style="padding: 12px 16px; text-align: center; border: 1px solid #334155;">Vencedor</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #ffffff;">
      <td style="padding: 12px 16px; font-weight: 600; color: #1e293b; border: 1px solid #e2e8f0;">R² (Coeficiente de Determinação)</td>
      <td style="padding: 12px 16px; text-align: center; color: #475569; border: 1px solid #e2e8f0;">0.67</td>
      <td style="padding: 12px 16px; text-align: center; font-weight: 600; color: #0f172a; border: 1px solid #e2e8f0;">0.90</td>
      <td style="padding: 12px 16px; text-align: center; color: #16a34a; font-weight: 600; border: 1px solid #e2e8f0;">+34.3%</td>
      <td style="padding: 12px 16px; text-align: center; border: 1px solid #e2e8f0;"><span style="background-color: #dcfce7; color: #15803d; padding: 4px 8px; border-radius: 6px; font-weight: 600; font-size: 12px;">RL Múltipla</span></td>
    </tr>
    <tr style="background-color: #f8fafc;">
      <td style="padding: 12px 16px; font-weight: 600; color: #1e293b; border: 1px solid #e2e8f0;">RMSE (Raiz do Erro Quadrático Médio)</td>
      <td style="padding: 12px 16px; text-align: center; color: #475569; border: 1px solid #e2e8f0;">3.20 un.</td>
      <td style="padding: 12px 16px; text-align: center; font-weight: 600; color: #0f172a; border: 1px solid #e2e8f0;">1.77 un.</td>
      <td style="padding: 12px 16px; text-align: center; color: #16a34a; font-weight: 600; border: 1px solid #e2e8f0;">-44.7%</td>
      <td style="padding: 12px 16px; text-align: center; border: 1px solid #e2e8f0;"><span style="background-color: #dcfce7; color: #15803d; padding: 4px 8px; border-radius: 6px; font-weight: 600; font-size: 12px;">RL Múltipla</span></td>
    </tr>
  </tbody>
</table>

Isso mesmo, o RMSE despencou quase pela metade ao introduzir o novo preditor `radio`! Esse resultado, em conjunto com um aumento significativo de $R^2$, nos diz que o modelo foi muito aprimorado pelas contribuições dos novos dados introduzidos para treinamento, explicando melhor a variação de $Y$ e reduzindo significativamente os erros de previsão.

Sob essa lógica, por que não apenas continuar adicionando parâmetros, como `jornal`?

### A adição da nova variável `jornal` e suas consequências

A seleção de atributos que serão incorporados ao treinamento de modelos regressivos é um tópico de vasta profundidade, visto que há diversas sutilezas que precisam ser analisadas de antemão. A princípio, podemos checar a *matriz de correlação*, a qual relaciona cada variável da nossa tabela, de dois a dois, com o respectivo valor de *correlação de Pearson* ($r$), o qual indica a força e direção da relação linear entre duas variáveis contínuas. $r$ igual a 0 aponta nenhum padrão linear reconhecido, próximo a 1 indica uma tendência de proporcionalidade direta, e próximo a -1, o inverso disso [3].

$$
\tag{2.3}
\text{Cor}(X, Y) = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2} \sqrt{\sum_{i=1}^{n}(y_i - \bar{y})^2}}
$$

In [65]:
# # Mostra a correlação de Pearson entre todos os preditores
matriz_corr = df.corr();

<table style="width: 100%; border-collapse: collapse; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); font-size: 13.5px; border: 1px solid #cbd5e1;">
  <thead>
    <tr style="background-color: #0f172a; color: #ffffff;">
      <th style="padding: 12px 16px; text-align: left; border: 1px solid #334155;">Variável</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">TV</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">Rádio</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">Jornal</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">Vendas</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 16px; font-weight: 600; color: #1e293b; border: 1px solid #e2e8f0;">TV</td>
      <td style="padding: 10px 14px; text-align: center; color: #94a3b8; background-color: #f8fafc; border: 1px solid #e2e8f0;">1.0</td>
      <td style="padding: 10px 14px; text-align: center; color: #475569; border: 1px solid #e2e8f0;">0.05</td>
      <td style="padding: 10px 14px; text-align: center; color: #475569; border: 1px solid #e2e8f0;">0.06</td>
      <td style="padding: 10px 14px; text-align: center; font-weight: bold; color: #1e293b; border: 1px solid #e2e8f0;">0.78</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 16px; font-weight: 600; color: #1e293b; border: 1px solid #e2e8f0;">Rádio</td>
      <td style="padding: 10px 14px; text-align: center; color: #475569; border: 1px solid #e2e8f0;">0.05</td>
      <td style="padding: 10px 14px; text-align: center; color: #94a3b8; background-color: #f8fafc; border: 1px solid #e2e8f0;">1.0</td>
      <td style="padding: 10px 14px; text-align: center; background-color: #fee2e2; color: #991b1b; font-weight: 600; border: 1px solid #fca5a5;">0.35</td>
      <td style="padding: 10px 14px; text-align: center; font-weight: bold; color: #1e293b; border: 1px solid #e2e8f0;">0.58</td>
    </tr>
    <tr style="background-color: #ffffff;">
      <td style="padding: 10px 16px; font-weight: 600; color: #1e293b; border: 1px solid #e2e8f0;">Jornal</td>
      <td style="padding: 10px 14px; text-align: center; color: #475569; border: 1px solid #e2e8f0;">0.06</td>
      <td style="padding: 10px 14px; text-align: center; background-color: #fee2e2; color: #991b1b; font-weight: 600; border: 1px solid #fca5a5;">0.35</td>
      <td style="padding: 10px 14px; text-align: center; color: #94a3b8; background-color: #f8fafc; border: 1px solid #e2e8f0;">1.0</td>
      <td style="padding: 10px 14px; text-align: center; background-color: #fef3c7; color: #92400e; font-weight: 600; border: 1px solid #fcd34d;">0.23</td>
    </tr>
    <tr style="background-color: #f1f5f9;">
      <td style="padding: 10px 16px; font-weight: 700; color: #0f172a; border: 1px solid #cbd5e1;">Vendas</td>
      <td style="padding: 10px 14px; text-align: center; font-weight: bold; color: #0f172a; border: 1px solid #cbd5e1;">0.78</td>
      <td style="padding: 10px 14px; text-align: center; font-weight: bold; color: #0f172a; border: 1px solid #cbd5e1;">0.58</td>
      <td style="padding: 10px 14px; text-align: center; background-color: #fef3c7; color: #92400e; font-weight: 700; border: 1px solid #fcd34d;">0.23</td>
      <td style="padding: 10px 14px; text-align: center; color: #94a3b8; background-color: #f8fafc; border: 1px solid #cbd5e1;">1.0</td>
    </tr>
  </tbody>
</table>

Com enfoque na análise de `jornal`, é válido destacar sua fraca correlação com `vendas` quando comparada àquela de outros meios, de 23%, além de que essa variável tem uma associação comprometedora com `radio`, de 35%! Traduzindo: mercados em que investe-se bem em rádio tendem a ter investimento similar nos jornais, e como `radio` tem um bom engajamento com `vendas`, a importância de `jornal` para prever `vendas` é "carregada" por `radio`. Esse fenômeno é chamado de colinearidade, em que o modelo não consegue separar as contribuições individuais das variáveis. 

Desse modo, uma variável que realmente tem influência sobre a resposta pode parecer insignificante simplesmente porque ela partilha de muita informação com outra no modelo. Ainda pior, os estimadores de coeficiente tornam-se instáveis, visto que adicionar ou remover um único ponto de dado ou uma variável do treinamento pode alterar drasticamente os demais coeficientes. Todos esses problemas não necessariamente se materializam em grandes erros ou declínio da precisão, mas fazem com que o modelo não enxergue a contribuição verdadeira de cada atributo [4].

De fato, a tabela a seguir mostra como a inclusão de `jornal` para o treinamento do modelo aumenta o $RMSE$ e reduz o $R^2$, mesmo que por pequeníssimas margens.

<table style="width: 100%; border-collapse: collapse; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); font-size: 13.5px; border: 1px solid #cbd5e1;">
  <thead>
    <tr style="background-color: #0f172a; color: #ffffff;">
      <th style="padding: 12px 16px; text-align: left; border: 1px solid #334155;">Modelo</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">&beta;&#770;<sub>0</sub></th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">&beta;&#770;<sub>1</sub> (TV)</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">&beta;&#770;<sub>2</sub> (Rádio)</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">&beta;&#770;<sub>3</sub> (Jornal)</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">R&sup2 (teste)</th>
      <th style="padding: 12px 14px; text-align: center; border: 1px solid #334155;">RMSE</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #ffffff;">
      <td style="padding: 11px 16px; font-weight: 600; color: #1e293b; border: 1px solid #e2e8f0;">RLM (TV + Rádio)</td>
      <td style="padding: 11px 14px; text-align: center; color: #334155; border: 1px solid #e2e8f0;">3.028</td>
      <td style="padding: 11px 14px; text-align: center; color: #334155; border: 1px solid #e2e8f0;">0.044</td>
      <td style="padding: 11px 14px; text-align: center; color: #334155; border: 1px solid #e2e8f0;">0.191</td>
      <td style="padding: 11px 14px; text-align: center; color: #94a3b8; border: 1px solid #e2e8f0;">&mdash;</td>
      <td style="padding: 11px 14px; text-align: center; font-weight: 600; color: #0f172a; border: 1px solid #e2e8f0;">0.900</td>
      <td style="padding: 11px 14px; text-align: center; font-weight: 600; color: #0f172a; border: 1px solid #e2e8f0;">1.771</td>
    </tr>
    <tr style="background-color: #f8fafc;">
      <td style="padding: 11px 16px; font-weight: 600; color: #1e293b; border: 1px solid #e2e8f0;">RLM (TV + Rádio + Jornal)</td>
      <td style="padding: 11px 14px; text-align: center; color: #334155; border: 1px solid #e2e8f0;">2.979</td>
      <td style="padding: 11px 14px; text-align: center; color: #334155; border: 1px solid #e2e8f0;">0.044</td>
      <td style="padding: 11px 14px; text-align: center; color: #334155; border: 1px solid #e2e8f0;">0.189</td>
      <td style="padding: 11px 14px; text-align: center; color: #334155; border: 1px solid #e2e8f0;">0.003</td>
      <td style="padding: 11px 14px; text-align: center; font-weight: 600; color: #0f172a; border: 1px solid #e2e8f0;">0.899</td>
      <td style="padding: 11px 14px; text-align: center; font-weight: 600; color: #0f172a; border: 1px solid #e2e8f0;">1.782</td>
    </tr>
  </tbody>
</table>

## A seleção de atributos, viés e variância

A investigação que fizemos acerca da variável `jornal` no último tópico tem um importante propósito didático e logrou êxito especialmente porque o número de variáveis era pequeno em comparação com o que é usualmente trabalhado. Mas realizar essa investigação em problemas reais seria imprático e tornaria exaustivo o processo de determinar quais variáveis serão encarregadas de treinar nosso modelo e em que grau o farão. Por curiosidade, pesquisadores da área da genética geralmente lidam com **milhares** de variáveis [5]. Como veremos, existem formas alternativas de ajuste que são capazes de aperfeiçoar a precisão e interpretabilidade do modelo.

### O dilema Viés-Variância 

Acerca de todos os tipos de modelo de aprendizado de máquina, permeia esse dilema conhecido em inglês como *Bias-Variance Trade-off*. Ele parte de duas métricas que são chaves para essa ciência:

- **Variância**: mede o quanto que $\hat{f}$ mudaria se usássemos diferentes dados para treinar o modelo. Ou seja, uma maior variância implica que as estimativas de coeficientes seriam substancialmente alteradas por pequenas mudanças nos dados de treino, o que não é o ideal. No geral, métodos estatísticos com mais variância tendem a ser mais flexíveis, isto é, com maior facilidade para se ajustar a diferentes padrões.

- **Viés**: afere o erro introduzido no modelo quando ele é aproximado à realidade. Ou seja, é quando um problema real - muito complexo - é simplificado, o que dificilmente caracteriza corretamente todos os fatores que influenciam um evento ou objeto. Nesse contexto, o MQM propõe uma relação linear entre os dados, quando na realidade é muito raro que isso de fato ocorra. Por exemplo, a aplicação dessa técninca a um problema cúbico (não linear) resulta em um alto viés, enquanto o problema das vendas visto anteriormente possui um baixo viés.

## Como selecionar os atributos bem pode nos guiar pelo dilema

- Desempenho: dada uma relação aproximadamente linear entre preditores e resposta, o modelo induzido via MQM terá um baixo viés. Se o número de obsrevações for muito maior do que o de variáveis independentes, isto é, n >> p, então também terá variância em um nível baixo, e assim o modelo terá um bom desempenho nos testes. Contudo, se n não for muito maior do que p, a variância pode ser problemática para o ajuste por MQM, resultando em *overfitting* (sobreajuste) e, por consequência, um resultado muito impreciso nos testes. Se p > n, então passa a haver infinitas soluções para MQM. Ao *encolher* ou *restringir* variáveis os coeficientes estimados, é possível reduzir drasticamente a variância ao custo de um pequeno aumento no viés, o que torna as previsões das respostas mais certeiras para os dados de teste.

- Interpretabilidade: Frequentemente parte dasvariáveis utilizadas em um modelo de RLM não são realmente associadas com a resposta. Ao incluir essas variáveis irrelevantes, abre-se espaço para uma complexidade desnecessária do modelo final. Quando elas são removidas (o coeficiente correspodnente torna-se 0), somos capaz de compreender melhor o comportamento do modelo - ele fica mais interpretáve. A essa abordagem chama-se seleção de atriburos, a qual pode assumir várias formas, como vamos explorar a seguir.


## Referências 

[1] https://www.geeksforgeeks.org/machine-learning/ml-linear-regression/

[2] https://statisticsfundamentals.com/simple-linear-regression/rmse/

[3] https://statisticsfundamentals.com/pearson-correlation/

[4] https://scienceinsights.org/what-is-collinearity-and-how-does-it-affect-regression/

[5] https://www.nature.com/articles/s43586-021-00056-9



<div style="width: 100%; text-align: center; margin-bottom: 20px;">
  <img src="images/rodape_institucional.png" alt="Rodapé" style="max-width: 100%; height: auto;">
</div>